# L5B15 Email/Outbound — Modeling

Alternative modeling approach on the Email/Outbound/Luggage/Sales L5B15 scoring events.

Reads from `data/processed/l5b15_email_outbound.parquet` — run `02_data_ingestion.ipynb` first.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

print("Imports OK")

In [ ]:
PROCESSED_FILE = Path("../data/processed/l5b15_email_outbound.parquet")
df = pd.read_parquet(PROCESSED_FILE)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Model versions: {df['modelVersion'].nunique()}  ({df['modelVersion'].value_counts().to_dict()})")
df.head(3)

## Feature preparation

In [ ]:
# Drop identifier and target-leakage columns
DROP_COLS = [
    "propensity",
    "pxInteractionID",
    "pxSubjectID",
    "modelExecutionID",
    "pyName",
    "pyChannel",
    "pyDirection",
    "pyGroup",
    "pyIssue",
    "TreatmentID",
    "modelPerformance",
    "modelEvidence",
    "modelTechnique",
]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()
y = df["propensity"].astype(float)

# Drop near-empty and constant columns
X = X.loc[:, X.isna().mean() < 0.95]
X = X.loc[:, X.nunique(dropna=True) > 1]

# Normalise missing-value markers
X = X.replace({"": np.nan, "NaN": np.nan, "nan": np.nan, "None": np.nan})

# Coerce to numeric where possible
for col in X.columns:
    try:
        X[col] = pd.to_numeric(X[col])
    except Exception:
        pass

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"Features: {X.shape[1]}  ({len(num_cols)} numeric, {len(cat_cols)} categorical)")

## Train / test split

> **Note:** Consider whether a random split is appropriate here.
> If records are ordered in time, a time-based split (earlier → train, later → test)
> better reflects production use. If multiple `modelVersion` values are present,
> consider stratifying or restricting to a single version.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

## Model

In [ ]:
# TODO: define and train your model here
# model = ...
# model.fit(X_train, y_train)
# pred = model.predict(X_test)

## Evaluation

In [ ]:
# Uncomment after training:
# print("RMSE: ", np.sqrt(mean_squared_error(y_test, pred)))
# print("MAE:  ", mean_absolute_error(y_test, pred))
# print("R²:   ", r2_score(y_test, pred))
# rho, _ = spearmanr(y_test, pred)
# print("Spearman ρ:", rho)

In [ ]:
# Actual vs predicted
# plt.scatter(y_test, pred, alpha=0.2)
# plt.xlabel("Actual propensity")
# plt.ylabel("Predicted propensity")
# plt.title("Actual vs predicted")
# plt.show()